# Project-Level Cumulative Spend Exploratory Analysis

This notebook examines cumulative burn at the project level across all customers. It intentionally does not assume that the project-level curve follows the contract-item-level Beta CDF shape. The goal is to characterize the data, understand sparsity and curve families, and identify reasonable model families for later validation.

## Dataset Profile

| Metric | Value |
| --- | --- |
| Input CSV | custpaydetails_project_clustered_cumulative_curves_allcustomers_2026-06-08-1000.csv |
| CSV rows parsed | 28,295 |
| Unique customer/projects | 3,417 |
| Customers present | Adams, CCD, CLV, Lincoln, UDOT |
| Train projects | 2,691 |
| Test projects | 726 |
| Median clusters/project | 5.0 |
| Projects with 1 cluster | 831 (24.32%) |
| Projects with <=6 clusters | 1,934 (56.60%) |
| Negative cluster rows | 346 |
| Projects with decreasing cumulative pct | 272 |

### Customer Coverage

| Customer | Rows | Projects |
| --- | --- | --- |
| Adams | 730 | 82 |
| CCD | 869 | 135 |
| CLV | 3 | 2 |
| Lincoln | 8,826 | 1,855 |
| UDOT | 17,867 | 1,343 |

### Missing Fields

| Field | Missing Rows | Share |
| --- | --- | --- |
| CUMULATIVEBURNPCT | 3,375 | 11.93% |

In [ ]:
import csv
from pathlib import Path
path = Path('custpaydetails_project_clustered_cumulative_curves_allcustomers_2026-06-08-1000.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Project Size, Duration, and Cluster Sparsity

Project-level curves are much more heterogeneous than contract-item curves. Many projects have very few observed payment clusters, which means a single smooth curve family cannot be evaluated fairly without stratifying by observation density.

| Cluster-count quantile | Clusters |
| --- | --- |
| 0 | 1.0 |
| 0.25 | 2.0 |
| 0.5 | 5.0 |
| 0.75 | 11.0 |
| 0.9 | 19.0 |
| 0.95 | 26.0 |
| 0.99 | 48.0 |
| 1 | 98.0 |







## Cumulative Spend Joint Distribution

The joint density shows the empirical relationship between elapsed project time and cumulative project spend. The diagonal is the pure linear reference. Because many projects have only a few clusters, the density includes many early/final jumps rather than smooth monthly progressions.

| Metric | Value |
| --- | --- |
| Project total burn median | 1,019,208.14 |
| Project total burn p90 | 6,277,380.71 |
| Project total burn p99 | 33,946,504.71 |
| Project modeled days median | 118.0 |
| Project modeled days p90 | 465.0 |
| Project modeled days p99 | 1,230.9 |
| Mean clipped cumulative pct | 65.01% |
| Median clipped cumulative pct | 77.36% |
| Pearson elapsed vs cumulative pct | 0.7525 |
| Spearman elapsed vs cumulative pct | 0.8140 |



## Conditional Percentile Bands and Candidate Curves

This view is descriptive, not a final model choice. It overlays empirical conditional bands with several candidate curve families: linear, monotone empirical/isotonic, logistic, Gompertz, and power curves. The empirical spread is wide, especially because sparse projects jump directly from low cumulative spend to completion.

| Elapsed bucket | Rows | Mean | Median | P10 | P25 | P75 | P90 | IQR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.00-0.05 | 264 | 9.44% | 3.72% | 0.13% | 0.62% | 7.81% | 25.65% | 7.19% |
| 0.05-0.10 | 968 | 13.09% | 7.86% | 0.83% | 3.62% | 15.02% | 31.68% | 11.39% |
| 0.10-0.15 | 1,230 | 22.80% | 16.10% | 2.43% | 7.89% | 29.83% | 53.95% | 21.95% |
| 0.15-0.20 | 1,283 | 31.96% | 25.27% | 4.78% | 12.96% | 46.48% | 72.78% | 33.51% |
| 0.20-0.25 | 1,337 | 41.37% | 36.77% | 8.37% | 19.91% | 61.17% | 84.30% | 41.26% |
| 0.25-0.30 | 1,314 | 49.22% | 45.85% | 13.08% | 27.65% | 73.14% | 89.58% | 45.50% |
| 0.30-0.35 | 1,216 | 57.17% | 58.23% | 19.12% | 35.16% | 82.26% | 92.61% | 47.09% |
| 0.35-0.40 | 1,181 | 63.17% | 66.00% | 24.71% | 42.43% | 87.31% | 95.48% | 44.87% |
| 0.40-0.45 | 1,124 | 68.07% | 73.68% | 32.46% | 49.36% | 90.26% | 96.98% | 40.89% |
| 0.45-0.50 | 1,029 | 73.04% | 79.08% | 40.20% | 57.96% | 92.44% | 97.70% | 34.48% |
| 0.50-0.55 | 1,035 | 77.61% | 85.36% | 44.31% | 65.24% | 94.97% | 98.15% | 29.74% |
| 0.55-0.60 | 916 | 81.42% | 89.18% | 50.99% | 72.09% | 95.98% | 98.81% | 23.89% |
| 0.60-0.65 | 843 | 84.27% | 91.69% | 58.24% | 76.56% | 97.11% | 99.14% | 20.54% |
| 0.65-0.70 | 844 | 87.17% | 93.85% | 66.05% | 81.50% | 98.12% | 99.51% | 16.62% |
| 0.70-0.75 | 749 | 88.45% | 94.57% | 67.98% | 85.61% | 98.50% | 99.70% | 12.89% |
| 0.75-0.80 | 689 | 92.17% | 96.11% | 80.25% | 90.55% | 98.95% | 99.81% | 8.40% |
| 0.80-0.85 | 609 | 92.73% | 97.24% | 82.38% | 92.06% | 99.43% | 99.92% | 7.37% |
| 0.85-0.90 | 557 | 93.75% | 98.49% | 83.89% | 94.46% | 99.70% | 99.95% | 5.24% |
| 0.90-0.95 | 513 | 94.32% | 98.86% | 88.80% | 95.59% | 99.81% | 99.99% | 4.22% |
| 0.95-1.00 | 1,954 | 99.18% | 100.00% | 99.56% | 100.00% | 100.00% | 100.00% | 0.00% |



## Model Family Screening

These are preliminary held-out point-level errors, not final production model results. They are useful for screening model families. The main question is whether the project-level shape looks linear, S-shaped, monotone empirical, power-law/front-loaded, or heterogeneous enough to require mixture models.

| Model family | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Empirical median curve | 0.1395 | 0.2129 | 0.0784 | 0.3661 | 0.0391 |
| Monotone empirical/isotonic curve | 0.1395 | 0.2129 | 0.0784 | 0.3661 | 0.0391 |
| Gompertz curve | 0.1486 | 0.2066 | 0.1059 | 0.3459 | 0.0251 |
| Logistic S-curve | 0.1486 | 0.2080 | 0.1105 | 0.3432 | 0.0319 |
| Power curve | 0.1637 | 0.2136 | 0.1406 | 0.3445 | 0.0121 |
| Linear reference | 0.1959 | 0.2605 | 0.1512 | 0.4335 | -0.1378 |

## Cluster Count Effects

Cluster count is not just a data-quality variable; it changes what curve shapes are observable. One-cluster projects only tell us the final point. Two- and three-cluster projects can look extremely front-loaded or back-loaded depending on when the first large payment occurs. Higher-cluster projects provide enough points to support smoother curve families.



## Duration and Size Effects

Duration and project size can plausibly change curve shape. Short projects may pay in a few bursts; large and long projects are more likely to have smoother cumulative progressions. These strata should be considered before choosing a single global project-level model.





## Empirical Curve Archetypes

The table classifies projects by interpolated spend position at 25%, 50%, and 75% elapsed time. This is a rough descriptive archetype analysis. It suggests whether a mixture/segmented model is more appropriate than a single global curve.

| Archetype | Projects | Rows |
| --- | --- | --- |
| approximately balanced | 272 | 3,583 |
| back-loaded | 101 | 732 |
| early-mid weighted | 954 | 13,326 |
| front-loaded | 440 | 5,282 |
| late-mid weighted | 159 | 1,895 |



## Assessment of Reasonable Model Families

The project-level distribution does not look like a simple reuse of the contract-item Beta CDF problem. The strongest immediate candidates are:

- **Empirical percentile bands**: best for exploratory reporting and threshold design because they expose the wide conditional spread.
- **Monotone empirical / isotonic curves**: a conservative global expected-position curve when the goal is calibration rather than parametric elegance.
- **Piecewise linear curves**: likely practical because project curves are often sparse and payment bursts create segment-like behavior.
- **Mixture models by cluster count, duration, size, and archetype**: probably necessary if the goal is good project-level prediction across all projects.
- **Logistic/Gompertz/Richards-type S-curves**: plausible for long, multi-cluster projects, but not for all projects. They should be tested only after excluding or separately handling sparse projects.
- **Linear baseline**: should remain a benchmark and may be competitive for some strata.

The immediate next analysis should separate sparse projects from dense projects, then evaluate model families per stratum. A single global parametric curve is unlikely to be adequate for all project-level behavior.

## Key Caveats

- This SQL intentionally included all projects with no minimum project spend or cluster-count filter, so the dataset mixes one-cluster, two-cluster, and dense projects.
- A project is an aggregation of many contract items with different start/end timings; the curve may reflect portfolio composition as much as construction progress.
- Corrections and negative postings can create non-monotonic raw cumulative percentages, while expected spend models are usually monotone smoothers.
- Point-level errors overweight projects with many clusters. Project-level validation should also compute one-project-one-weight metrics.
- The project-level denominator is final historical project spend. Live use will need a clear operational denominator, such as authorized budget or forecast final cost.